# Bài 20: Phát hiện và So khớp đặc trưng (Feature Detection and Matching)

Đây là phần mở đầu của Phần 2: thị giác máy tính 3D cổ điển, nơi các thuật toán ngày càng làm việc trên *nhiều* ảnh của một cảnh &mdash; các khung hình video, hoặc nhiều camera &mdash; thay vì chỉ một ảnh. Vấn đề đầu tiên nảy sinh: hai ảnh của cùng một cảnh hiếm khi đặt cùng một nội dung tại cùng một vị trí điểm ảnh, vì phép xoay, tỉ lệ và góc nhìn đều làm mọi thứ dịch chuyển. Bài này tiến dần tới so khớp dựa trên đặc trưng (feature-based matching), vốn chịu được chính những thay đổi đó: đầu tiên là **các bộ phát hiện góc (corner detectors)** (Harris, Shi-Tomasi) tìm ra các điểm đặc biệt, lặp lại được, sau đó là **SIFT**, bổ sung tính bất biến với tỉ lệ và một bộ mô tả (descriptor) có thể so khớp giữa hai ảnh trông khác nhau của cùng một cảnh.

In [ ]:
# Download the sample images used in this lesson, if not already present
import os, urllib.request

for _f in ['building.png']:
    _path = f"../img/{_f}"
    if not os.path.exists(_path):
        os.makedirs("../img", exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/sbirchfield/sbirchfield.github.io/main/cvintro/img/{_f}", _path)

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Tensor cấu trúc: điều gì tạo nên một điểm tốt để so khớp?

Tại mỗi điểm ảnh, xây dựng **tensor cấu trúc (structure tensor)** $2\times2$ từ cửa sổ cục bộ các gradient (Bài 12):

$$M = \sum_{\text{window}} \begin{bmatrix}I_x^2 & I_xI_y \\ I_xI_y & I_y^2\end{bmatrix}$$

Điều này hẳn trông quen thuộc: đó chính xác là ma trận hiệp phương sai mô-men ở Bài 6, nhưng được xây dựng từ gradient thay vì tọa độ điểm ảnh. Các giá trị riêng $\lambda_1 \ge \lambda_2$ của nó mô tả cấu trúc cường độ cục bộ:

- cả hai đều nhỏ: vùng **phẳng**, gradient yếu theo mọi hướng
- một lớn, một nhỏ: **biên**, gradient mạnh theo phương vuông góc với biên, yếu dọc theo biên
- cả hai đều lớn: **góc**, gradient mạnh theo mọi hướng &mdash; chính là loại điểm có thể được định vị lại chính xác trong một ảnh khác

In [ ]:
img = np.zeros((100, 100), dtype=np.uint8)
cv2.rectangle(img, (20, 20), (80, 80), 200, -1)
cv2.rectangle(img, (40, 40), (60, 60), 100, -1)
img_f = img.astype(np.float64)

Ix = cv2.Sobel(img_f, cv2.CV_64F, 1, 0, ksize=3)
Iy = cv2.Sobel(img_f, cv2.CV_64F, 0, 1, ksize=3)

window = 5
Sxx = cv2.boxFilter(Ix * Ix, -1, (window, window))
Syy = cv2.boxFilter(Iy * Iy, -1, (window, window))
Sxy = cv2.boxFilter(Ix * Iy, -1, (window, window))

plt.imshow(img, cmap='gray')
plt.title('Test image: corners, edges, and flat regions')
plt.axis('off')
plt.show()

Ảnh động này trượt cùng một cửa sổ 5x5 quanh ảnh này (có thêm một chút nhiễu điểm ảnh để phân bố gradient thực sự là một đám mây điểm) &mdash; phân bố gradient và elip riêng của tensor cấu trúc cập nhật trực tiếp, và việc phân loại (phẳng / biên / góc) suy ra trực tiếp từ hai giá trị riêng, đúng theo quy tắc ở trên.

<img src="../animations/structure_tensor_sweep_demo.gif" alt="Ảnh động cửa sổ trượt tính tensor cấu trúc cục bộ khi di chuyển quanh ảnh, cho thấy phân bố gradient và elip riêng cập nhật trực tiếp kèm phân loại phẳng/biên/góc" style="display: block; margin: 0 auto; max-width: 680px;">

## Bộ phát hiện góc Harris

Tính trực tiếp các giá trị riêng ở mọi nơi hơi tốn kém, nên <a href="../references.html#harris-stephens-1988">Harris và Stephens (1988)</a> đề xuất một đại lượng thay thế rẻ hơn chỉ dùng $\det(M)$ và $\text{trace}(M)$ (có thể tính mà không cần tìm giá trị riêng):

$$R = \det(M) - k \cdot \text{trace}(M)^2, \qquad k \approx 0.04$$

$R$ lớn và dương ở góc, âm ở biên, và gần 0 ở vùng phẳng.

In [ ]:
k = 0.04
det_M = Sxx * Syy - Sxy**2
trace_M = Sxx + Syy
harris_response = det_M - k * trace_M**2

harris_cv = cv2.cornerHarris(img_f.astype(np.float32), blockSize=window, ksize=3, k=k)
print('correlation with cv2.cornerHarris:', np.corrcoef(harris_response.ravel(), harris_cv.ravel())[0, 1])

fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
axes[0].imshow(img, cmap='gray')
axes[0].set_title('Image')
im = axes[1].imshow(harris_response, cmap='coolwarm')
axes[1].set_title('Harris response R\n(red=corner, blue=edge)')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

## Shi-Tomasi: "Good Features to Track"

<a href="../references.html#shi-tomasi-1994">Shi và Tomasi (1994)</a> lập luận nên dùng trực tiếp **giá trị riêng nhỏ nhất** của $M$ thay vì đại lượng thay thế định thức/vết của Harris:

$$R_{\text{ST}} = \min(\lambda_1, \lambda_2)$$

Một điểm chỉ là góc mạnh nếu *cả hai* giá trị riêng đều lớn &mdash; giá trị nhỏ nhất lớn chính là điều kiện đó, và (khác với $R$ của Harris) nó có ý nghĩa trực tiếp, dễ diễn giải: nó tỉ lệ với độ chính xác theo dõi trong trường hợp xấu nhất theo bất kỳ hướng nào.

In [ ]:
discriminant = np.sqrt(np.clip((trace_M / 2)**2 - det_M, 0, None))
lambda_min = trace_M / 2 - discriminant

eigval_cv = cv2.cornerMinEigenVal(img_f.astype(np.float32), blockSize=window, ksize=3)
print('correlation with cv2.cornerMinEigenVal:', np.corrcoef(lambda_min.ravel(), eigval_cv.ravel())[0, 1])

corners = cv2.goodFeaturesToTrack(img_f.astype(np.uint8), maxCorners=20, qualityLevel=0.1, minDistance=5)

plt.imshow(img, cmap='gray')
for pt in corners[:, 0]:
    plt.scatter(*pt, c='red', s=40, marker='+')
plt.title('cv2.goodFeaturesToTrack (Shi-Tomasi)')
plt.axis('off')
plt.show()

## Vấn đề tỉ lệ

Harris và Shi-Tomasi đều hoạt động ở một kích thước cửa sổ cố định duy nhất. Phóng to hoặc thu nhỏ cùng một góc, và nó có thể không còn trông giống góc ở tỉ lệ cố định đó (một góc nhọn trở thành một đường cong thoải khi thu nhỏ đủ nhiều) &mdash; các bộ phát hiện này không bất biến với tỉ lệ. **SIFT** (Scale-Invariant Feature Transform — biến đổi đặc trưng bất biến tỉ lệ) (<a href="../references.html#lowe-2004">Lowe, 1999/2004</a><span class="landmark-paper">&#9733;</span>) khắc phục điều này bằng cách tìm kiếm điểm đặc trưng trên toàn bộ một không gian tỉ lệ (scale-space), không chỉ một kích thước cửa sổ.

## SIFT: cực trị không gian tỉ lệ và bộ mô tả

Quy trình của SIFT, nhìn tổng quát:

1. Xây dựng một **không gian tỉ lệ Hiệu-của-Gauss (DoG)** &mdash; chính là phép xấp xỉ DoG cho toán tử Laplace ở Bài 13, tính ở nhiều mức làm mờ.
2. Tìm điểm đặc trưng dưới dạng cực trị cục bộ của phản hồi DoG theo cả không gian *lẫn* tỉ lệ &mdash; một điểm là cực đại so với 26 lân cận của nó (gồm 8 lân cận ở tỉ lệ hiện tại *và* 9 lân cận ở mỗi mức tỉ lệ phía trên và phía dưới).
3. Gán cho mỗi điểm đặc trưng một hướng chủ đạo từ histogram gradient cục bộ, để bộ mô tả có thể được làm bất biến với phép xoay.
4. Xây dựng một bộ mô tả 128 chiều từ các histogram hướng gradient trong một lưới $4\times4$ các vùng con quanh điểm đặc trưng.

Kết quả: một điểm đặc trưng có vị trí, tỉ lệ, hướng, và một vector mô tả được thiết kế để rất giống nhau giữa hai ảnh ngay cả khi có phép xoay, co giãn và thay đổi chiếu sáng ở mức vừa phải.

Ảnh động này minh họa phép kiểm tra cực trị 26 lân cận trên một không gian tỉ lệ Hiệu-của-Gauss tổng hợp với hai đốm kích thước khác nhau. Bên trái, chấm đánh dấu vị trí (x, y) của điểm ảnh ứng viên khi nó trượt qua ảnh ở các tỉ lệ khác nhau. Bên phải, ba mảng cho thấy 26 lân cận của ứng viên (8 ở cùng tỉ lệ cộng với các mảng 3x3 tương ứng ở một mức tỉ lệ phía trên và phía dưới). Một ô màu cam đánh dấu giá trị nhỏ nhất trong tất cả 27 giá trị. Khi cực trị lệch khỏi tâm, điểm ảnh ứng viên không phải điểm đặc trưng. Ô chỉ chuyển sang màu xanh lục khi cực trị rơi đúng vào điểm ảnh ứng viên: tâm của đốm nhỏ hơn ở sigma nhỏ hơn, và tâm của đốm lớn hơn (bán kính gấp đôi) ở một quãng tám (octave) sau đó &mdash; chính là cơ chế chọn tỉ lệ giúp khắc phục vấn đề tỉ lệ nêu ở trên.

<img src="../animations/sift_scalespace_demo.gif" alt="Ảnh động phép kiểm tra cực trị không gian tỉ lệ SIFT: một điểm ảnh ứng viên trượt qua ảnh trong khi một ô theo dõi giá trị nhỏ nhất trong 27 giá trị lân cận không gian+tỉ lệ, chuyển xanh lục đúng lúc nó rơi vào chính ứng viên tại tâm và tỉ lệ khớp của mỗi đốm" style="display: block; margin: 0 auto; max-width: 680px;">

In [ ]:
textured = cv2.imread('../img/building.png', cv2.IMREAD_GRAYSCALE)
textured = cv2.cvtColor(textured, cv2.COLOR_BGR2RGB)

sift = cv2.SIFT_create()
keypoints, descriptors = sift.detectAndCompute(textured, None)
print(f'found {len(keypoints)} keypoints, each with a {descriptors.shape[1]}-dim descriptor')

keypoint_vis = cv2.drawKeypoints(textured, keypoints, None,
                                  flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(textured)
axes[0].set_title('Original')
axes[1].imshow(keypoint_vis)
axes[1].set_title('SIFT keypoints (circle size = scale, line = orientation)')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

<p class="image-attribution">Ảnh của <a href="https://unsplash.com/@dawsonlovell" target="_blank" rel="noopener">Dawson Lovell</a> trên <a href="https://unsplash.com/photos/flatiron-building-in-toronto-W_MUqtuHwyY" target="_blank" rel="noopener">Unsplash</a></p>

## So khớp khi có phép xoay và thay đổi tỉ lệ

Ta xoay và thu nhỏ ảnh bằng một phép biến đổi *đã biết*, phát hiện đặc trưng SIFT độc lập trên cả hai ảnh, rồi so khớp các bộ mô tả bằng tìm kiếm láng giềng gần nhất. **Phép kiểm tra tỉ số của Lowe (Lowe's ratio test)** chỉ giữ một cặp khớp nếu ứng viên tốt nhất gần hơn đáng kể so với ứng viên tốt *thứ hai* &mdash; một cách đơn giản, hiệu quả để loại bỏ các cặp khớp mơ hồ.

In [ ]:
M = cv2.getRotationMatrix2D((150, 150), angle=35, scale=0.7)
rotated_scaled = cv2.warpAffine(textured, M, (300, 300))

kp1, des1 = sift.detectAndCompute(textured, None)
kp2, des2 = sift.detectAndCompute(rotated_scaled, None)

bf = cv2.BFMatcher()
raw_matches = bf.knnMatch(des1, des2, k=2)
good_matches = [m for m, n in raw_matches if m.distance < 0.75 * n.distance]

print(f'keypoints: {len(kp1)} (original), {len(kp2)} (rotated + scaled)')
print(f'good matches after Lowe ratio test: {len(good_matches)} / {len(raw_matches)} candidate pairs')

### Có bao nhiêu cặp khớp thực sự đúng?

Vì ta biết chính xác phép biến đổi `M` dùng để tạo ảnh thứ hai, ta có thể kiểm tra trực tiếp: ánh xạ mỗi điểm đặc trưng đã khớp qua `M` có rơi gần vị trí nó thực sự được khớp không?

In [ ]:
correct = 0
for m in good_matches:
    p1 = np.array(kp1[m.queryIdx].pt + (1,))
    p2 = np.array(kp2[m.trainIdx].pt)
    predicted = M @ p1
    if np.linalg.norm(predicted - p2) < 3:
        correct += 1

print(f'geometrically correct matches: {correct} / {len(good_matches)} ({100 * correct / len(good_matches):.0f}%)')

match_vis = cv2.drawMatches(textured, kp1, rotated_scaled, kp2, good_matches, None,
                             flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
plt.figure(figsize=(11, 5))
plt.imshow(match_vis)
plt.title(f'{len(good_matches)} matches across a 35-degree rotation + 0.7x scale change')
plt.axis('off')
plt.show()

Gần như tất cả các cặp vượt qua phép kiểm tra tỉ số đều là các tương ứng thực sự đúng, bất chấp phép xoay và thay đổi tỉ lệ &mdash; điều mà một bộ phát hiện cố định ở một tỉ lệ và một hướng đơn giản là không thể khôi phục. Đây chính là điều khiến các đặc trưng kiểu SIFT trở thành cách tiếp cận kinh điển cho các nhiệm vụ như ghép ảnh, nhận dạng đối tượng và định vị bằng thị giác.

### Bài tập

1. Tăng góc xoay lên 90 độ và tỉ lệ xuống 0.3. Số cặp khớp tốt và phần trăm đúng về mặt hình học có giữ vững, hay suy giảm? Đến mức nào thì SIFT bắt đầu gặp khó khăn?
2. Thử `cv2.ORB_create()` (một phương án thay thế SIFT nhanh hơn nhiều, miễn phí, dùng bộ mô tả nhị phân) với `cv2.NORM_HAMMING` trong `BFMatcher` thay vì chuẩn L2 mặc định. So sánh số cặp khớp tốt và so sánh định tính tốc độ bằng `%timeit` trên `detectAndCompute`.
3. Hạ ngưỡng kiểm tra tỉ số từ `0.75` xuống `0.6`. Số cặp khớp tốt và tỉ lệ đúng về mặt hình học thay đổi thế nào? Điều này cho bạn biết gì về vai trò của ngưỡng trong sự đánh đổi độ chính xác/độ phủ (precision/recall)?